 Задача 1

 --------

 - Разложить $x^7 - 1$ на множители (должно быть 3) над полем $F_2$

 - Построить из их комбинаций $6$ порождающих полиномов

 - Определить характеристики полученных кодов

 - Определить, что это за коды?

 Решение задачи 1

In [3]:
from itertools import product
from IPython.display import Markdown


def poly_to_int(exponents):  # функция перевода полинома в целое число
    value = 0
    for power in exponents:  # перебираем степени полинома
        value |= 1 << power  # устанавливаем бит в позиции степени
    return value


def poly_degree(poly):  # функция нахождения степени полинома
    if poly == 0:  # если полином равен 0, то возвращаем -1
        return -1
    return poly.bit_length() - 1  # возвращаем степень полинома


def poly_mul(a, b):  # функция умножения полиномов
    res = 0
    len_b = poly_degree(b)
    for shift in range(len_b + 1):
        if (b >> shift) & 1:
            res ^= a << shift
    return res


def find_3_multies(a):  # функция поиска трех множителей для разложения полинома a
    for num_1 in range(
        2, 2 ** (poly_degree(a))
    ):  # перебираем возможные значения первого множителя
        for num_2 in range(
            2, 2 ** (poly_degree(a))
        ):  # перебираем возможные значения второго множителя
            for num_3 in range(
                2, 2 ** (poly_degree(a))
            ):  # перебираем возможные значения третьего множителя
                if (
                    poly_mul(num_1, poly_mul(num_2, num_3)) == a
                ):  # если произведение трех множителей равно a
                    return num_1, num_2, num_3  # возвращаем найденные множители


def poly_to_str(poly):  # функция перевода полинома в строку
    if poly == 0:  # если полином равен 0, то возвращаем "0"
        return "0"
    terms = []
    for power in range(poly_degree(poly), -1, -1):  # перебираем степени полинома
        if (poly >> power) & 1:  # если полином равен 1
            if power == 0:  # если степень полинома равна 0, то добавляем "1"
                terms.append("1")
            elif power == 1:  # если степень полинома равна 1, то добавляем "x"
                terms.append("x")
            else:  # если степень полинома больше 1, то добавляем "x^power"
                terms.append(f"x^{power}")
    return " + ".join(terms)  # возвращаем строку из термов


def encode_cyclic(message, generator):  # функция кодирования циклического кода
    message_poly = 0  # полином сообщения
    for i, bit in enumerate(message):  # перебираем биты сообщения
        if bit:  # если бит равен 1
            message_poly |= 1 << i  # устанавливаем бит в позиции i
    code_poly = poly_mul(
        message_poly, generator
    )  # умножаем полином сообщения на генератор
    return tuple(
        (code_poly >> i) & 1 for i in range(7)
    )  # возвращаем кортеж из бит кодового слова


def all_codewords_for_g(
    generator,
):  # функция нахождения всех кодовых слов для генератора
    r = poly_degree(generator)  # степень генератора
    k = 7 - r  # размерность кода
    words = [
        encode_cyclic(message, generator) for message in product([0, 1], repeat=k)
    ]  # перебираем все возможные сообщения
    return sorted(
        set(words)
    )  # возвращаем отсортированный список уникальных кодовых слов


def weight(vector):  # функция нахождения веса вектора
    return sum(vector)  # возвращаем сумму вектора


def min_distance(
    codewords,
):  # функция нахождения минимального расстояния между кодовыми словами
    nonzero = [word for word in codewords if any(word)]  # список ненулевых кодовых слов
    return (
        min(weight(word) for word in nonzero) if nonzero else 0
    )  # возвращаем минимальный вес ненулевых кодовых слов


def identify_code(k, d):
    if k == 6 and d == 2:
        return "код с проверкой на четность (7,6,2)"
    if k == 4 and d == 3:
        return "код, эквивалентный коду Хэмминга (7,4,3)"
    if k == 3 and d == 4:
        return "симплексный код (7,3,4), эквивалентные циклические реализации"
    if k == 1 and d == 7:
        return "повторный код (7,1,7)"
    return "двоичный циклический код длины 7"


out = ""
out += "Разложение над $F2$: <br>"
poly_a = int("10000001", 2)

# 1. Данный полином:
print(f"Дан полином: {bin(poly_a)[2:]} ({poly_to_str(poly_a)})")
print(f"степени - {poly_degree(poly_a)}")

# 2. Находим разложение:
f1, f2, f3 = find_3_multies(poly_a)

# 3. Выводим множители в бинарном виде и в виде полинома:
print("Множители, найденные разложением:")
print(f"f1 = {bin(f1)[2:]}: ({poly_to_str(f1)})")
print(f"f2 = {bin(f2)[2:]}: ({poly_to_str(f2)})")
print(f"f3 = {bin(f3)[2:]}: ({poly_to_str(f3)})")

# 4. Последовательность перемножения:
ab = poly_mul(f1, f2)
abc = poly_mul(ab, f3)

print("\nПоследовательность перемножения:")
print(f"({bin(f1)[2:]}) * ({bin(f2)[2:]}) = {bin(ab)[2:]} ({poly_to_str(ab)})")
print(f"({bin(ab)[2:]}) * ({bin(f3)[2:]}) = {bin(abc)[2:]} ({poly_to_str(abc)})")
print(
    f"Итого: ({poly_to_str(f1)}) * ({poly_to_str(f2)}) * ({poly_to_str(f3)}) = ({poly_to_str(abc)}) = ({poly_to_str(poly_a)})"
)
print(f"Сходится: {abc == poly_a}")
print("x^7 - 1 = x^7 + 1 = (x + 1)(x^3 + x + 1)(x^3 + x^2 + 1)")

# 6 нетривиальных комбинаций множителей (без 1 и без x^7 + 1)
generators = [
    f1,
    f2,
    f3,
    poly_mul(f1, f2),
    poly_mul(f1, f3),
    poly_mul(f2, f3),
]

print("6 порождающих полиномов и характеристики кодов:\n")
for index, g in enumerate(generators, start=1):  # перебираем генераторы
    r = poly_degree(g)  # степень генератора
    k = 7 - r  # размерность кода
    codewords = all_codewords_for_g(g)  # все кодовые слова для генератора
    d = min_distance(codewords)  # минимальное расстояние между кодовыми словами
    code_type = identify_code(k, d)  # тип кода

    print(f"{index}: g(x) = {poly_to_str(g)}")
    print(f"   deg(g) = {r}, параметры: (n,k,d) = (7,{k},{d}), |C| = 2^{k} = {2**k}")
    print(f"   Тип: {code_type}")
    print("-" * 40)

Дан полином: 10000001 (x^7 + 1)
степени - 7
Множители, найденные разложением:
f1 = 11: (x + 1)
f2 = 1011: (x^3 + x + 1)
f3 = 1101: (x^3 + x^2 + 1)

Последовательность перемножения:
(11) * (1011) = 11101 (x^4 + x^3 + x^2 + 1)
(11101) * (1101) = 10000001 (x^7 + 1)
Итого: (x + 1) * (x^3 + x + 1) * (x^3 + x^2 + 1) = (x^7 + 1) = (x^7 + 1)
Сходится: True
x^7 - 1 = x^7 + 1 = (x + 1)(x^3 + x + 1)(x^3 + x^2 + 1)
6 порождающих полиномов и характеристики кодов:

1: g(x) = x + 1
   deg(g) = 1, параметры: (n,k,d) = (7,6,2), |C| = 2^6 = 64
   Тип: код с проверкой на четность (7,6,2)
----------------------------------------
2: g(x) = x^3 + x + 1
   deg(g) = 3, параметры: (n,k,d) = (7,4,3), |C| = 2^4 = 16
   Тип: код, эквивалентный коду Хэмминга (7,4,3)
----------------------------------------
3: g(x) = x^3 + x^2 + 1
   deg(g) = 3, параметры: (n,k,d) = (7,4,3), |C| = 2^4 = 16
   Тип: код, эквивалентный коду Хэмминга (7,4,3)
----------------------------------------
4: g(x) = x^4 + x^3 + x^2 + 1
   deg(